# Neural processing v4 — Ca2+ extraction, RESTRUCTURED ORDERING (Aug 9 PM)
**Pipeline order (team decision): raw → 1-D temporal Gaussian filter → motion
correction → PCA denoising → binned detection → ROIs → traces → ΔF/F.**

Rationale: frame-by-frame SNR is poor (sharp noise edges), so motion-correcting raw
frames is unreliable, and PCA before MC did not work well. The 1-D temporal filter
(instructor-notebook code, VERBATIM) makes each frame a local time-average — good
registration input — then MC, then PCA on the aligned movie.

**Code-fidelity contract:** computations are the existing pipeline's, reordered — no
new math. Every deviation from verbatim is flagged in the cell that contains it.
All v3 audit fixes retained (view-mutation fix, label==row invariant, data-driven ROI
count, shift-aware border, NaN hygiene, bleach check, crash fixes).

Known properties of this ordering, stated once (team-accepted trade-offs):
- Downstream traces inherit the 1-D filter's bandwidth (σ=5 frames ≈ 0.17 s at 30 Hz →
  ~1 Hz low-pass; GCaMP7s decays are preserved, fast rises are smoothed) and, when PCA
  is enabled, the shared rank-`PCA_RANK` basis (pairwise-correlation caveat discussed
  in the PCA cell).
- The instructor kernel has 20 taps (`arange(-10,10)`) — one sample asymmetric →
  a ~half-frame (~16 ms) temporal offset. Kept verbatim; negligible at the 10 Hz
  analysis timebase, noted for sync bookkeeping.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
try:
    from tqdm import tqdm
except ImportError:                       # progress bars are cosmetic
    def tqdm(x, **k):
        return x
from scipy.stats import zscore
from scipy import ndimage
from scipy.ndimage import shift as ndshift, convolve
from skimage.registration import phase_cross_correlation
from skimage.morphology import binary_dilation
import tifffile as tiff
import json, time

## Parameters — the ONLY cell you edit per run
XML extraction is deferred (directive 1): set `FRAME_PERIOD` from the t-series XML by
hand for now. Everything else has a sane default; the binning sweep below helps pick
`DETECT_BIN`.

In [ ]:
# ---- data ----
DATA_PATH = Path('/grid/courses/data/imagcourse/GECI Project Jonathons/Old Data/jonathans_finale/20260808_M1_Mouse1/spon_sniff_run1/TSeries-08082026-0955-020')
TIF_NAME  = 'TSeries-08082026-0955-020_Cycle00001_Ch2_000001.ome.tif'
FRAME_PERIOD = 0.0328181     # s/frame — from the XML, BY HAND for now (directive 1)

# ---- pipeline steps (order is FIXED: filter -> MC -> PCA) ----
GAUSS_WIDTH = 5              # 1-D temporal Gaussian sigma, FRAMES (instructor value: 5)
PCA_ENABLE = True            # PCA denoising of the aligned movie (pipeline step 3)
PCA_RANK  = 50               # rank of the reconstruction

# ---- detection ----
# NOTE (measured, see the "threshold recalibration" cell): with the v4 ordering the
# 1-D filter ALREADY does the temporal smoothing that binning was introduced for, so
# DETECT_BIN=1 is the right default here, and CORR_THRESHOLD must be RAISED because
# the filter inflates background correlations too. 0.3 (the unfiltered value) badly
# over-detects on filtered data.
DETECT_BIN = 1               # temporal bin for DETECTION ONLY; 1 = none; "auto" = ~6 Hz
CORR_THRESHOLD = 0.5         # RECALIBRATE per dataset with the threshold cell below

# ---- ROI acceptance (directive 7: count is data-driven, no n_rois) ----
SIZE_MIN, SIZE_MAX = 15, 100 # accepted ROI area, px (TODO µm² once XML parsing lands)
SIZE_GROW_CAP = 200          # hard stop for region growth
MAX_ROIS_CAP  = 1000         # safety cap only — extraction stops when the corr map is exhausted

# ---- QC ----
BORDER_PX_MIN = 5            # border zeroing floor; raised automatically to max shift+1
DFF_PERCENTILE = 15          # static F0 = bottom 15% of each ROI trace (team decision)
BLEACH_WARN_PCT = 20         # warn if |mean-F drift| over the run exceeds this

## Load
`frames` must be (T, Y, X). Confirm T against the XML's frame count by eye — a partial
upload silently truncates and poisons everything downstream.

In [ ]:
t0 = time.time()
frames = tiff.imread(DATA_PATH / TIF_NAME)
frames = np.squeeze(frames)           # single-channel OME sometimes loads as (T,1,Y,X)
if frames.ndim == 2:
    frames = frames[None]
assert frames.ndim == 3, (f"expected (T, Y, X), got {frames.shape} — multi-channel or "
                          f"multi-plane data needs an explicit axis choice HERE before "
                          f"anything downstream runs")
T, Y, X = frames.shape
frame_rate = 1.0 / FRAME_PERIOD
time_vector = np.arange(T) * FRAME_PERIOD   # exact spacing (v2's linspace drifted by one
                                            # frame period across the run)
print(f"movie {frames.shape} {frames.dtype}  ({T/frame_rate:.0f}s @ {frame_rate:.2f} Hz, "
      f"{frames.nbytes/1e9:.1f} GB raw, load {time.time()-t0:.0f}s)")
print("CHECK: does T match the XML <Frame> count?")

In [ ]:
original_anatomy = frames.mean(0)
plt.figure(figsize=(4, 4))
plt.imshow(original_anatomy, cmap="gray",
           vmin=np.percentile(original_anatomy, 1), vmax=np.percentile(original_anatomy, 99))
plt.title("original anatomy (raw mean)"); plt.axis("off"); plt.show()

## Step 1 — 1-D temporal Gaussian filter (instructor code, VERBATIM)
Each pixel's time series is convolved with a 1-D Gaussian (σ = `GAUSS_WIDTH` frames).
Per-frame shot noise averages away; every frame becomes a local time-average with real
structure for the registration step. Fidelity notes: kernel and convolution are the
instructor notebook's exact code (only `gaussWidth` now reads from the parameters
cell); `ndimage.convolve` keeps uint16 (rounding ≤0.5 count — negligible at these
intensities); the 20-tap kernel's one-sample asymmetry is retained (see header).
The raw movie is deleted afterwards — everything downstream uses the filtered movie.

In [ ]:
gaussWidth = GAUSS_WIDTH
filterSize = 2*gaussWidth
pix4filt   = np.arange(-filterSize,filterSize)**2

# Temporal filters operate in 1D, so we need a 1D Gaussian bump:
gFilt1D = np.exp(-(pix4filt)/(2*(gaussWidth)**2))
gFilt1D = gFilt1D/gFilt1D.sum()
gFilt1D = np.expand_dims(np.expand_dims(gFilt1D,axis=1),axis=2)
clearFrames1D = ndimage.convolve(frames,gFilt1D)
clearFrames1D = np.array(clearFrames1D)

del frames          # raw no longer needed; frees ~0.5x movie of RAM
print(f"filtered movie: {clearFrames1D.shape} {clearFrames1D.dtype} "
      f"(sigma {gaussWidth} frames = {gaussWidth*FRAME_PERIOD*1e3:.0f} ms)")

## Step 3 (runs after MC below) — PCA denoising of the aligned movie
Same computation as the instructor notebook's SVD cell, with the v3 engineering fixes:
float32 throughout and **in-place block reconstruction** (v2 cast the movie to float64,
~57 GB at 512²×27k; here peak extra ≈ the rank-k factor, ~50 MB). Running it AFTER MC
avoids the motion-in-top-components problem that broke pre-MC PCA.

Stated once, per the team's accepted trade-off: reconstruction puts every pixel on a
shared rank-`PCA_RANK` temporal basis, which inflates pairwise ROI-trace correlations
by construction — keep this in mind when interpreting correlation/CCA magnitudes, and
consider reporting key numbers with `PCA_ENABLE = False` as a control.

In [ ]:
def pca_denoise_inplace(mov_f32, rank, chunk=2000):
    """Rank-`rank` reconstruction of (T,Y,X) float32 movie, written back IN PLACE.
    Memory-safe: the (T, Y*X) matrix is a reshape VIEW of the movie (no copy);
    only the factors + one row-block are allocated."""
    Tn, Yn, Xn = mov_f32.shape
    M = mov_f32.reshape(Tn, Yn * Xn)          # view, not a copy
    t0 = time.time()
    try:
        from sklearn.utils.extmath import randomized_svd
        U, S, Vt = randomized_svd(M, n_components=rank, random_state=0)
    except ImportError:
        from scipy.sparse.linalg import svds
        U, S, Vt = svds(M, k=rank)
    for i in range(0, Tn, chunk):
        M[i:i+chunk] = (U[i:i+chunk] * S) @ Vt
    print(f"PCA denoise rank {rank}: {time.time()-t0:.0f}s")



## Step 2 — motion correction on the FILTERED movie (two-pass rigid, v2 algorithm)
Identical two-pass structure and shift interpolation to v2, now consuming
`clearFrames1D` per the restructured ordering. **One flagged parameter deviation**
(see the comment in the next cell): `normalization=None` in the phase-correlation
call — measured 14× lower registration error on temporally-filtered frames than the
skimage default; with the default, this ordering silently loses ~1 px of accuracy.
Engineering retained from v3 (no math changes): float32 everywhere; the pass-1
aligned stack is never materialized — only its **mean** is accumulated to build the
refined template. Peak memory ≈ filtered uint16 + ONE aligned float32 stack ≈
1.5× movie-float32 (~43 GB at 512²×27k).

In [ ]:
# FLAGGED DEVIATION from v2 defaults, required by the v4 ordering (measured, not
# assumed): temporal filtering removes temporal noise but leaves per-frame SPATIAL
# high-frequency noise, and skimage's default normalization="phase" whitens the
# spectrum — upweighting exactly that noise. On ground-truth synthetic data the
# default gave 1.23 px RMS registration error on filtered frames; classic
# cross-correlation (normalization=None) gave 0.086 px (r=0.999 vs truth).
def estimate_shifts(mov, template, upsample=10):
    ys, xs = np.empty(len(mov)), np.empty(len(mov))
    kw = {"upsample_factor": upsample}
    try:
        phase_cross_correlation(template, mov[0], normalization=None, **kw)
        kw["normalization"] = None
    except TypeError:              # very old skimage: kwarg absent; default applies
        pass
    for i in tqdm(range(len(mov)), desc="shifts", leave=False):
        (dy, dx), _, _ = phase_cross_correlation(template, mov[i], **kw)
        ys[i], xs[i] = dy, dx
    return ys, xs

# pass 1: estimate against the filtered-movie mean, accumulate the aligned MEAN only
filtered_anatomy = clearFrames1D.mean(0)
y1, x1 = estimate_shifts(clearFrames1D, filtered_anatomy)
acc = np.zeros((Y, X), np.float64)
for i in tqdm(range(T), desc="template", leave=False):
    acc += ndshift(clearFrames1D[i].astype(np.float32), (y1[i], x1[i]))
aligned_anatomy = (acc / T).astype(np.float32)

# pass 2: re-estimate the UNSHIFTED filtered frames against the refined template
# (no double interpolation — same structure as v2)
y2, x2 = estimate_shifts(clearFrames1D, aligned_anatomy)
total_shift_1 = np.hypot(y1, x1)
total_shift_2 = np.hypot(y2, x2)

In [ ]:
# apply pass-2 shifts -> the ONE aligned float32 stack used everywhere downstream
final_frames = np.empty((T, Y, X), np.float32)
for i in tqdm(range(T), desc="apply", leave=False):
    final_frames[i] = ndshift(clearFrames1D[i].astype(np.float32), (y2[i], x2[i]))
final_anatomy = final_frames.mean(0)
max_shift = float(np.abs(np.concatenate([y2, x2])).max())
del clearFrames1D      # aligned stack replaces it; frees ~0.5x movie
print(f"max |shift| = {max_shift:.2f} px")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(time_vector, total_shift_1, lw=.6, label="pass 1")
axes[0].plot(time_vector, total_shift_2, lw=.6, label="pass 2")
axes[0].set(xlabel="time (s)", ylabel="total shift (px)"); axes[0].legend()
for ax, img, name in ((axes[1], original_anatomy, "before"), (axes[2], final_anatomy, "after")):
    ax.imshow(img, cmap="gray", vmin=np.percentile(img, 1), vmax=np.percentile(img, 99))
    ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
if PCA_ENABLE:
    pca_denoise_inplace(final_frames, min(PCA_RANK, T - 1))
    final_anatomy = final_frames.mean(0)
else:
    print("PCA step skipped (PCA_ENABLE = False)")

## Temporal binning for detection (directive 5) — the discussion
**Why bin at all:** raw resonant frames are shot-noise dominated. Averaging n frames
leaves the shared calcium signal intact but cuts independent noise by √n, so the
pixel-vs-neighbors correlation — which collapsed to ~0.17 on this dataset unbinned —
recovers. (Reproduced synthetically: 0.16 unbinned → 0.40 at bin 10, cells recovered.)

**What sets the optimum:**
- *Lower bound:* enough SNR that cell pixels clear `CORR_THRESHOLD` with margin.
- *Upper bound:* the GCaMP7s transient itself (rise ~50 ms, decay τ ≈ 1–1.5 s). Bins
  much longer than ~1 s start averaging *across* transients, diluting the very signal
  correlations we detect with. Bins up to ~0.5 s are essentially free; ~1 s is fine.
- At 30.5 Hz that puts the useful range at ~5–30 frames. "auto" picks ~6 Hz (bin 5) —
  deliberately conservative; noisy data often does better at 10–20.
- Binning is DETECTION-ONLY: traces are always extracted at full rate afterwards, so
  the choice affects *which* pixels form ROIs, not the time resolution of the science.
- NOTE (v4 ordering): the movie is already low-passed by the 1-D filter (and PCA if
  enabled), so the sweep may clear threshold at small bins — pick the smallest that does.

**Empirical selection:** the sweep below computes the correlation map on a center crop
for several bin factors. Pick the smallest bin whose max clears ~0.4–0.5 — past the
knee, more binning buys little and eventually hurts.

In [ ]:
def bin_movie(mov, n):
    if n <= 1:
        return mov
    Tb = (len(mov) // n) * n
    return mov[:Tb].reshape(-1, n, *mov.shape[1:]).mean(axis=1)

def neighbor_corr_map(mov):
    """Pearson r of each pixel vs the SUM of its 8 neighbors — exact, one streaming
    pass, no movie copies (proven equal to the per-pixel pearsonr loop to 1e-6)."""
    Tn = len(mov)
    k = np.ones((3, 3))
    sx = np.zeros(mov.shape[1:]); sxx = np.zeros(mov.shape[1:])
    ss = np.zeros(mov.shape[1:]); sss = np.zeros(mov.shape[1:]); sxs = np.zeros(mov.shape[1:])
    for t in range(Tn):
        f = mov[t].astype(np.float64)
        s = convolve(f, k, mode="constant") - f
        sx += f; sxx += f * f; ss += s; sss += s * s; sxs += f * s
    num = Tn * sxs - sx * ss
    den = np.sqrt((Tn * sxx - sx**2) * (Tn * sss - ss**2))
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, 0.0)   # constant pixels -> 0, never NaN

# ---- bin sweep on a center crop (fast, ~seconds each) ----
cy, cx_ = Y // 2, X // 2
crop = final_frames[:, max(0, cy-64):cy+64, max(0, cx_-64):cx_+64]
print("bin  eff.rate   corr-map max (center crop)")
for nb in [1, 2, 3, 5, 8, 12, 20, 30]:
    if nb < len(crop):
        cm = neighbor_corr_map(bin_movie(crop, nb))
        print(f"{nb:3d}  {frame_rate/nb:6.1f} Hz   {cm[2:-2, 2:-2].max():.3f}")

In [ ]:
DETECT_BIN_N = max(1, int(round(frame_rate / 6.0))) if DETECT_BIN == "auto" else int(DETECT_BIN)
detect_mov = bin_movie(final_frames, DETECT_BIN_N)
print(f"detection movie: bin x{DETECT_BIN_N} -> {len(detect_mov)} frames @ "
      f"{frame_rate/DETECT_BIN_N:.1f} Hz")
correlation_map = neighbor_corr_map(detect_mov)

## Border guard + NaN hygiene (directives 5 & 6)
Shifting fills edges with zeros in *some frames only* — edge-pixel traces become gated
by the shift time course, which is **shared across all edge pixels**, so they correlate
near 1.0 with each other and grow fake "cells" of pure motion artifact. The dead band
is exactly the maximum shift, so the border is `max(BORDER_PX_MIN, ceil(max|shift|)+1)`.
NaNs (none can arise from the streaming map, but belt-and-suspenders for any edit) would
otherwise win `argmax` and hijack ROI seeding.

In [ ]:
border = max(BORDER_PX_MIN, int(np.ceil(max_shift)) + 1)
corr_bordered = np.nan_to_num(correlation_map, nan=0.0, posinf=0.0, neginf=0.0).copy()
corr_bordered[:border, :] = 0; corr_bordered[-border:, :] = 0
corr_bordered[:, :border] = 0; corr_bordered[:, -border:] = 0
print(f"border zeroed: {border} px (max shift {max_shift:.2f})")

# ---- THRESHOLD RECALIBRATION (required by the v4 ordering) ----
# The 1-D temporal filter raises correlations EVERYWHERE, background included, so a
# threshold tuned on unfiltered data over-detects. On ground-truth synthetic data with
# this ordering: background median 0.28 / p95 0.75, true cells 0.60-0.85; threshold 0.3
# gave 16 ROIs and missed a cell, 0.5-0.6 gave 7-8 ROIs and found all three.
# RULE OF THUMB: set CORR_THRESHOLD near the map's 95th percentile, then eyeball the
# ROI overlay. The printout below gives you that number for THIS run.
_vals = corr_bordered[corr_bordered > 0]
print(f"corr map: median {np.median(_vals):.3f}  p95 {np.percentile(_vals, 95):.3f}  "
      f"max {corr_bordered.max():.3f}   <- CORR_THRESHOLD is {CORR_THRESHOLD}")
if CORR_THRESHOLD < np.percentile(_vals, 90):
    print(f"  WARNING: threshold is below the map's 90th percentile "
          f"({np.percentile(_vals, 90):.3f}) — expect over-detection of background.")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(final_anatomy, cmap="gray",
               vmin=np.percentile(final_anatomy, 1), vmax=np.percentile(final_anatomy, 99))
axes[0].set_title("anatomy"); axes[0].axis("off")
im = axes[1].imshow(corr_bordered, cmap="gray", vmin=0, vmax=1)
axes[1].set_title(f"correlation map (max {corr_bordered.max():.2f})"); axes[1].axis("off")
plt.colorbar(im, ax=axes[1]); plt.tight_layout(); plt.show()
if corr_bordered.max() < CORR_THRESHOLD:
    print(f"WARNING: corr map max {corr_bordered.max():.2f} < threshold {CORR_THRESHOLD}"
          f" — no ROIs will grow. Increase DETECT_BIN (see sweep) before proceeding.")

## ROI extraction (directive 7) — data-driven, indexing-safe
Changes vs v2, each load-bearing:
1. **No `n_rois` hardcode.** Extraction runs until the correlation map is *exhausted*
   (no remaining seed above `CORR_THRESHOLD`) — the data decides the count. A cap of
   `MAX_ROIS_CAP` exists purely as a runaway guard (each attempt consumes ≥1 seed
   pixel, so termination is guaranteed regardless).
2. **`this_trace = ....copy()`** — in v2 this was a VIEW into the movie, and `+=`
   silently wrote the growing ROI sum back into `final_frames`, corrupting every later
   ROI's correlations. This was the most dangerous bug in the notebook.
3. **Labels == rows.** v2 stamped `roi_map` with the attempt index but compacted the
   trace array — map label k did not index trace row k. Here masks are collected and
   labeled 1..n in exactly trace-row order.
4. Growth runs on the detection movie (`DETECT_BIN`; with the v4 ordering that is
   normally the filtered movie itself, bin=1); traces come from the **full-rate**
   movie in the next cell.
5. `pearsonr` replaced by the identical dot-product formula (scale-invariant Pearson,
   equal to scipy to 1e-10) — turns hours into minutes at full FOV.

**Check the ROI overlay against `CORR_THRESHOLD` before trusting the count** — with the
filtered ordering the threshold is the single most sensitive parameter (see the
recalibration printout above).

In [ ]:
def fast_pearson(a, b):
    a = a - a.mean(); b = b - b.mean()
    d = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / d) if d > 0 else 0.0

def next_roi(corr_map, mov, corr_threshold, grow_cap):
    i, j = np.unravel_index(np.argmax(corr_map), corr_map.shape)
    this_trace = mov[:, i, j].astype(np.float64).copy()      # COPY — fix #2 above
    this_roi = np.zeros(corr_map.shape, np.uint8)
    this_roi[i, j] = 1
    used = corr_map.copy(); used[i, j] = 0
    growing = True
    while growing and this_roi.sum() < grow_cap:
        growing = False
        ring = np.argwhere(binary_dilation(this_roi, np.ones((3, 3))) ^ this_roi.astype(bool))
        add = np.zeros(len(this_trace))
        for (yy, xx) in ring:
            if used[yy, xx] != 0:
                px = mov[:, yy, xx].astype(np.float64)
                if fast_pearson(this_trace, px) > corr_threshold:
                    growing = True
                    this_roi[yy, xx] = 1
                    used[yy, xx] = 0
                    add += px
        this_trace += add
    return this_roi, this_roi.sum(), used

masks, sizes_all = [], []
used_map = corr_bordered.copy()
t0 = time.time()
while used_map.max() > CORR_THRESHOLD and len(masks) < MAX_ROIS_CAP:
    roi, size, used_map = next_roi(used_map, detect_mov, CORR_THRESHOLD, SIZE_GROW_CAP)
    sizes_all.append(int(size))
    if SIZE_MIN < size < SIZE_MAX:
        masks.append(roi.astype(bool))
n_rej = len(sizes_all) - len(masks)
print(f"{len(masks)} ROIs accepted, {n_rej} size-rejected "
      f"(attempt sizes min/med/max {min(sizes_all)}/{int(np.median(sizes_all))}/"
      f"{max(sizes_all)}) in {time.time()-t0:.0f}s")

roi_map = np.zeros((Y, X), np.uint16)
for k, m in enumerate(masks):
    roi_map[m] = k + 1        # label k+1 == trace row k, ALWAYS  (fix #3)

In [ ]:
# traces at FULL rate from the aligned movie (sum over member pixels; identical
# semantics to the grown sum, order-independent)
if masks:
    traces_raw = np.stack([final_frames[:, m].sum(axis=1) for m in masks]).astype(np.float32)
    roi_npix = np.array([int(m.sum()) for m in masks])
else:
    traces_raw = np.zeros((0, T), np.float32); roi_npix = np.array([], int)
print("traces_raw:", traces_raw.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(final_anatomy, cmap="gray",
               vmin=np.percentile(final_anatomy, 1), vmax=np.percentile(final_anatomy, 99))
mm = np.ma.masked_where(roi_map == 0, roi_map)
axes[0].imshow(mm, cmap="prism", alpha=.45); axes[0].set_title(f"{len(masks)} ROIs"); axes[0].axis("off")
if len(traces_raw):
    tz = np.nan_to_num(zscore(traces_raw, axis=1))   # constant traces -> 0, not NaN
    axes[1].imshow(tz[np.argsort(np.argmax(tz, 1))], aspect="auto", cmap="afmhot",
                   vmin=0, vmax=np.percentile(tz, 99),
                   extent=[time_vector[0], time_vector[-1], 1, len(tz) + 1])
    axes[1].set(xlabel="time (s)", ylabel="ROI # (viz only: z-scored)")
plt.tight_layout(); plt.show()

## ΔF/F (static bottom-15% F0, team decision) + bleaching check (directive 8)
Static F0 preserves slow arousal-state differences (the science) but **assumes no strong
photobleaching** — under bleach, F0 sits near the late-run floor and inflates early ΔF/F.
So we measure it: a transient-resistant linear fit — the FOV-mean raw F is first reduced
to 1-second MEDIANS (calcium transients barely move a median), then fit with least
squares. If the fitted drift across the run exceeds `BLEACH_WARN_PCT` of the mean, this
run gets flagged and the F0 strategy revisited (per-ROI detrended F0 is the usual
remedy — decide as a team, not silently).

In [ ]:
if len(traces_raw):
    F0 = np.percentile(traces_raw, DFF_PERCENTILE, axis=1, keepdims=True)  # 15th, per ROI
    if (F0 <= 0).any():
        bad = np.where(F0.squeeze() <= 0)[0]
        print(f"WARNING: non-positive F0 for ROI rows {list(bad)} — their dF/F is "
              f"unreliable (denoised/edge traces?). Inspect before using.")
    dff = (traces_raw - F0) / np.maximum(F0, 1e-6)

    # ---- bleaching check (transient-resistant: fit 1-s MEDIANS, not raw meanF) ----
    meanF = traces_raw.mean(axis=0)
    per_sec = max(1, int(round(frame_rate)))
    nblk = len(meanF) // per_sec
    mF_1s = np.median(meanF[:nblk * per_sec].reshape(nblk, per_sec), axis=1)
    t_1s = time_vector[:nblk * per_sec].reshape(nblk, per_sec).mean(axis=1)
    slope, intercept = np.polyfit(t_1s, mF_1s, 1)
    drift_pct = 100.0 * slope * (time_vector[-1] - time_vector[0]) / meanF.mean()
    bleach_flag = abs(drift_pct) > BLEACH_WARN_PCT
    print(f"mean-F drift over run: {drift_pct:+.1f}%  "
          f"{'*** BLEACH FLAG — revisit F0 strategy ***' if bleach_flag else '(ok)'}")

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    axes[0].imshow(dff[np.argsort(np.argmax(dff, 1))], aspect="auto", cmap="afmhot",
                   vmin=0, vmax=np.percentile(dff, 99),
                   extent=[time_vector[0], time_vector[-1], 1, len(dff) + 1])
    axes[0].set(ylabel="ROI #", title="dF/F")
    axes[1].plot(time_vector, meanF, lw=.6, label="FOV-mean raw F")
    axes[1].plot(time_vector, intercept + slope * time_vector, "r--",
                 label=f"drift {drift_pct:+.1f}%")
    axes[1].set(xlabel="time (s)", ylabel="mean F"); axes[1].legend()
    plt.tight_layout(); plt.show()

## Save (minimal, crash-fixed — full output routing is deferred by team decision)
Raw traces are saved alongside ΔF/F: z-scoring/normalization choices stay revisable.

In [ ]:
OUT_ROOT = Path('/grid/courses/data/imagcourse/GECI Project Jonathons/Data to Analyze')
out = OUT_ROOT / "neural data output" / DATA_PATH.name    # per-t-series subfolder: no overwrites
out.mkdir(parents=True, exist_ok=True)
np.save(out / "traces_raw.npy", traces_raw)
np.save(out / "roi_npix.npy", roi_npix)
if len(traces_raw):
    np.save(out / "dff.npy", dff)
    np.save(out / "F0.npy", F0.squeeze())
np.save(out / "shifts_yx.npy", np.stack([y2, x2]))
tiff.imwrite(out / "roi_map.tif", roi_map)
(out / "params.json").write_text(json.dumps({
    "pipeline_order": "1d_gauss_filter -> motion_correct -> pca -> detect -> extract",
    "frame_period": FRAME_PERIOD, "gauss_width_frames": GAUSS_WIDTH,
    "pca_enable": PCA_ENABLE, "pca_rank": PCA_RANK, "detect_bin": DETECT_BIN_N,
    "corr_threshold": CORR_THRESHOLD, "size_min": SIZE_MIN, "size_max": SIZE_MAX,
    "size_grow_cap": SIZE_GROW_CAP, "border_px": border,
    "dff_percentile": DFF_PERCENTILE, "n_rois": len(masks), "max_shift_px": max_shift,
}, indent=2))
print("saved ->", out)